11_SOTA_and_diagnostics.py
Journal-revision experiments (all numbers are real, computed here):

  (1) SOTA baselines: LightGBM and CatBoost trained with the SAME leakage-safe
      pipeline and the SAME 5-fold split as the in-house XGBoost, so the
      comparison in the paper is fair.
  (2) Failure-case analysis: the development-set listings with the largest blend
      residuals, with their key characteristics.
  (3) Feature-family inventory from the engineered matrix.

Writes outputs/sota_results.txt and outputs/failure_cases.txt


In [13]:
# Notebook compatibility helper
import os
os.environ['PYTHONWARNINGS'] = 'ignore'  # also silences warnings from n_jobs=-1 joblib subprocesses
from pathlib import Path
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

In [14]:
import json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
OUT = Path('outputs')
RS, K = 42, 5
train = pd.read_parquet(OUT / 'train_local.parquet')
TARGET, ID = 'blocked_days_Q1_2026', 'id'
y = train[TARGET].astype(float).values
X = train.drop(columns=[TARGET, ID]).reset_index(drop=True)

cat_all = X.select_dtypes(exclude='number').columns.tolist()
card = {c: X[c].nunique(dropna=False) for c in cat_all}
cat_high = [c for c, n in card.items() if n > 15]
cat_low = [c for c, n in card.items() if n <= 15]
num_cols = X.select_dtypes(include='number').columns.tolist()

In [15]:
class KFoldTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols, n_splits=5, smoothing=20.0, random_state=42):
        self.cols = cols; self.n_splits = n_splits
        self.smoothing = smoothing; self.random_state = random_state

    def _smap(self, x, yy):
        st = pd.DataFrame({'c': x, 'y': yy}).groupby('c')['y'].agg(['mean', 'count'])
        return ((st['count'] * st['mean'] + self.smoothing * self.gm_)
                / (st['count'] + self.smoothing)).to_dict()

    def fit(self, X, y):
        y = np.asarray(y, float); self.gm_ = float(y.mean())
        self.maps_ = {c: self._smap(X[c].astype(str).fillna('__nan__'), y) for c in self.cols}
        return self

    def transform(self, X):
        Xo = X.copy()
        for c in self.cols:
            Xo[c] = (X[c].astype(str).fillna('__nan__').map(self.maps_[c])
                       .fillna(self.gm_).astype('float32'))
        return Xo

    def fit_transform(self, X, y=None, **kw):
        y = np.asarray(y, float); self.gm_ = float(y.mean())
        Xo = X.copy()
        for c in self.cols:
            Xo[c] = np.full(len(X), self.gm_, dtype='float32')
        kf = KFold(self.n_splits, shuffle=True, random_state=self.random_state)
        for tr, va in kf.split(X):
            for c in self.cols:
                m = self._smap(X[c].astype(str).fillna('__nan__').iloc[tr], y[tr])
                Xo.iloc[va, Xo.columns.get_loc(c)] = (
                    X[c].astype(str).fillna('__nan__').iloc[va].map(m)
                       .fillna(self.gm_).astype('float32').values)
        self.maps_ = {c: self._smap(X[c].astype(str).fillna('__nan__'), y) for c in self.cols}
        return Xo


In [16]:
def make_pp():
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
        ('low', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')),
                          ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_low),
        ('high', Pipeline([('te', KFoldTargetEncoder(cat_high, 5, 20, RS))]), cat_high),
    ])


## Hyperparameter Tuning for LightGBM & CatBoost

Previously both models used **fixed, untuned** hyperparameters. This adds the same kind of
fold-level-early-stopping random search already used for XGBoost in `08_XGBoost_v2.ipynb` /
`09_XGBoost_v3.ipynb`: for each candidate config, run the identical 5-fold CV split, carve a
10% inner-validation slice inside each fold's training portion, fit with early stopping
watching that slice, and average the resulting fold MSEs. The winning config per model family
is then reused — still with early stopping — for the final OOF / local-test predictions below,
so the reported iteration count actually matches what the search selected.

In [18]:
# Load local test set
test_local_df = pd.read_parquet(OUT / 'test_local.parquet')
X_local_test = test_local_df.drop(columns=[TARGET, ID]).reindex(columns=X.columns).reset_index(drop=True)
y_local_test = test_local_df[TARGET].astype(float).values

lines = ['SOTA BASELINES (5-fold CV, same split as XGBoost, TUNED)',
         '=' * 70, f'{"Model":<12}{"MSE":>10}{"MAE":>8}{"R2":>8}{"MSE_std":>10}{"sec":>8}']
for model_name, info in search_results.items():
    best_cfg, ctor = info['best_cfg'], info['ctor']
    t0 = time.time()
    final = evaluate_gbm(model_name, ctor, best_cfg, return_oof=True, return_models=True)
    sec = time.time() - t0
    oof = final['oof']
    mse, mae, r2 = mean_squared_error(y, oof), mean_absolute_error(y, oof), r2_score(y, oof)
    lines.append(f'{model_name:<12}{mse:>10.2f}{mae:>8.2f}{r2:>8.3f}{final["mse_std"]:>10.2f}{sec:>8.1f}')
    print(lines[-1])
    np.save(OUT / f'oof_{model_name}.npy', oof)

    # Local test prediction: average across the 5 fold models (each already early-stopped)
    test_local_pred = np.zeros(len(X_local_test))
    for m, pp in final['models']:
        Xtl = pp.transform(X_local_test)
        test_local_pred += np.clip(m.predict(Xtl), 0, 90)
    test_local_pred /= len(final['models'])

    local_test_mse = mean_squared_error(y_local_test, test_local_pred)
    np.save(OUT / f'test_local_pred_{model_name}.npy', test_local_pred)
    print(f'  Local Test MSE: {local_test_mse:.3f} | Saved test_local_pred_{model_name}.npy')

rep = '\n'.join(lines)
(OUT / 'sota_results.txt').write_text(rep, encoding='utf-8')
print('\nsaved sota_results.txt')

LightGBM        296.49   10.20   0.529      2.20    45.6
  Local Test MSE: 293.103 | Saved test_local_pred_LightGBM.npy
CatBoost        301.36   10.42   0.522      3.49    73.0
  Local Test MSE: 298.496 | Saved test_local_pred_CatBoost.npy

saved sota_results.txt
